In [23]:
# Section 1: 安装依赖
# 用途：安装本 notebook 所需依赖

!pip install -q numpy pandas sounddevice soundfile faster-whisper transformers accelerate sentencepiece ipywidgets widgetsnbextension jupyterlab_widgets

In [24]:
!pip install -q pyttsx3

In [25]:
# Section 2: 导入依赖
# 用途：导入本项目所需模块

import os
import re
import gc
import json
import time
import numpy as np
import pandas as pd
import sounddevice as sd
import soundfile as sf
import torch
import pyttsx3

from typing import Dict, Any, Optional, Tuple, List
from collections import deque
from faster_whisper import WhisperModel
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    StoppingCriteria,
    StoppingCriteriaList,
)

In [26]:
# Section 3: schema / validator / executor
# 用途：定义标准动作格式、合法性检查、模拟执行设备动作

COMMAND_SCHEMA = {
    "light": {
        "actions": {
            "turn_on": None,
            "turn_off": None,
            "set_brightness": (0, 100),
            "rgb_cycle": None,
        }
    },
    "curtain": {
        "actions": {
            "open": None,
            "close": None,
            "set_position": (0, 100),
        }
    },
    "window": {
        "actions": {
            "open": None,
            "close": None,
            "set_position": (0, 100),
        }
    },
    "ac": {
        "actions": {
            "turn_on": None,
            "turn_off": None,
            "set_temperature": (16, 30),
        }
    },
}

INVALID_COMMAND = {
    "device": "unknown",
    "action": "invalid",
    "value": None,
}

def validate_command(cmd: Dict[str, Any]) -> Tuple[bool, str]:
    required_keys = {"device", "action", "value"}

    if not isinstance(cmd, dict):
        return False, "command_not_dict"

    if set(cmd.keys()) != required_keys:
        return False, "invalid_keys"

    device = cmd["device"]
    action = cmd["action"]
    value = cmd["value"]

    if device == "unknown" and action == "invalid" and value is None:
        return False, "unrecognized_command"

    if device not in COMMAND_SCHEMA:
        return False, "invalid_device"

    valid_actions = COMMAND_SCHEMA[device]["actions"]
    if action not in valid_actions:
        return False, "invalid_action_for_device"

    value_rule = valid_actions[action]

    if value_rule is None:
        if value is not None:
            return False, "value_must_be_null"
        return True, "ok"

    if not isinstance(value, int):
        return False, "value_must_be_int"

    min_v, max_v = value_rule
    if not (min_v <= value <= max_v):
        return False, "value_out_of_range"

    return True, "ok"

def execute_command(cmd: Dict[str, Any]) -> str:
    device = cmd["device"]
    action = cmd["action"]
    value = cmd["value"]

    if device == "light":
        if action == "turn_on":
            return "LIGHT -> ON"
        if action == "turn_off":
            return "LIGHT -> OFF"
        if action == "set_brightness":
            return f"LIGHT -> BRIGHTNESS {value}%"
        if action == "rgb_cycle":
            return "LIGHT -> RGB CYCLE"

    if device == "curtain":
        if action == "open":
            return "CURTAIN -> OPEN"
        if action == "close":
            return "CURTAIN -> CLOSE"
        if action == "set_position":
            return f"CURTAIN -> POSITION {value}%"

    if device == "window":
        if action == "open":
            return "WINDOW -> OPEN"
        if action == "close":
            return "WINDOW -> CLOSE"
        if action == "set_position":
            return f"WINDOW -> POSITION {value}%"

    if device == "ac":
        if action == "turn_on":
            return "AC -> ON"
        if action == "turn_off":
            return "AC -> OFF"
        if action == "set_temperature":
            return f"AC -> TEMPERATURE {value}C"

    return "NO ACTION EXECUTED"

def build_execution_reply(cmd: Dict[str, Any]) -> str:
    device = cmd["device"]
    action = cmd["action"]
    value = cmd["value"]

    if device == "light" and action == "turn_on":
        return "Okay, I turned on the light."
    if device == "light" and action == "turn_off":
        return "Okay, I turned off the light."
    if device == "light" and action == "set_brightness":
        return f"Okay, I set the light brightness to {value} percent."
    if device == "light" and action == "rgb_cycle":
        return "Okay, I started the RGB cycle mode."

    if device == "curtain" and action == "open":
        return "Okay, I opened the curtain."
    if device == "curtain" and action == "close":
        return "Okay, I closed the curtain."
    if device == "curtain" and action == "set_position":
        return f"Okay, I set the curtain to {value} percent."

    if device == "window" and action == "open":
        return "Okay, I opened the window."
    if device == "window" and action == "close":
        return "Okay, I closed the window."
    if device == "window" and action == "set_position":
        return f"Okay, I set the window to {value} percent."

    if device == "ac" and action == "turn_on":
        return "Okay, I turned on the air conditioner."
    if device == "ac" and action == "turn_off":
        return "Okay, I turned off the air conditioner."
    if device == "ac" and action == "set_temperature":
        return f"Okay, I set the air conditioner to {value} degrees."

    return "Okay, the command was executed."

print("Schema / validator / executor loaded.")

Schema / validator / executor loaded.


In [27]:
# Section 4: assistant 名字与预筛选逻辑
# 用途：先轻量判断一条 STT 文本里是否包含 "Nova"

ASSISTANT_NAME = "nova"

ASSISTANT_NAME_VARIANTS = [
    "nova", "nava", "no va", "noba", "noa", "nove", "novia",
]

def contains_assistant_name(text: str) -> bool:
    text = text.strip().lower()
    return any(name in text for name in ASSISTANT_NAME_VARIANTS)

print("Assistant name filter loaded.")
print("Assistant name:", ASSISTANT_NAME)
print("Variants:", ASSISTANT_NAME_VARIANTS)

Assistant name filter loaded.
Assistant name: nova
Variants: ['nova', 'nava', 'no va', 'noba', 'noa', 'nove', 'novia']


In [28]:
# Section 5: 统一多任务 LLM parser
# 用途：让模型直接输出四类之一：
# 1) direct_command
# 2) needs_clarification
# 3) general_qa
# 4) invalid

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Loading unified multi-task LLM...")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


UNIFIED_SYSTEM_PROMPT = """
Return exactly one JSON object and nothing else.

Allowed outputs:

{"type":"direct_command","device":"light|curtain|window|ac","action":"turn_on|turn_off|set_brightness|rgb_cycle|open|close|set_position|set_temperature","value":null_or_int}

{"type":"needs_clarification","question":"...","options":["...","..."]}

{"type":"general_qa","answer":"..."}

{"type":"invalid"}

Rules:
- Named device + clear action → direct_command
- Indirect/comfort/mood/feeling request → needs_clarification
- Non-device question → general_qa
- No meaningful request → invalid
- Output JSON only. No extra text.

Examples:
Input: Nova, turn on the light.
Output: {"type":"direct_command","device":"light","action":"turn_on","value":null}

Input: Nova, set the AC to 24 degrees.
Output: {"type":"direct_command","device":"ac","action":"set_temperature","value":24}

Input: Nova, I feel cold.
Output: {"type":"needs_clarification","question":"Would you like me to close the window or raise the AC temperature?","options":["close_window","raise_ac_temperature"]}

Input: Nova, it's a bit dark.
Output: {"type":"needs_clarification","question":"Would you like me to turn on the light or open the curtain?","options":["turn_on_light","open_curtain"]}

Input: Nova, fuck this light.
Output: {"type":"needs_clarification","question":"Would you like me to turn off the light or dim it?","options":["turn_off_light","dim_light"]}

Input: Nova, make this room lively.
Output: {"type":"needs_clarification","question":"Would you like me to turn on the RGB cycle or open the curtain?","options":["rgb_cycle","open_curtain"]}

Input: Nova, how do I eat an apple?
Output: {"type":"general_qa","answer":"Wash it first, then eat it."}

Input: Nova, can I still eat this dish after a night in the fridge?
Output: {"type":"general_qa","answer":"Yes, most cooked food is safe for up to 3-4 days in the fridge."}

Input: Hello.
Output: {"type":"invalid"}
""".strip()


FOLLOWUP_RESOLUTION_SYSTEM_PROMPT = """
You are resolving the user's reply to a previous clarification question.

Return exactly one JSON object and nothing else.

Allowed output types:

1. direct_command
Use:
{"type":"direct_command","device":"...","action":"...","value":...}

Allowed device values:
- "light"
- "curtain"
- "window"
- "ac"

Allowed action values:
- "turn_on"
- "turn_off"
- "set_brightness"
- "rgb_cycle"
- "open"
- "close"
- "set_position"
- "set_temperature"

2. invalid
Use:
{"type":"invalid"}

Rules:
- Use the previous user request, the clarification question, the available options, and the current reply
- If the current reply clearly selects one option, return direct_command
- If the reply is unclear, return invalid
- No explanation
- No markdown
- No extra text
""".strip()


def extract_first_json_object(text: str) -> Optional[str]:
    start = text.find("{")
    if start == -1:
        return None

    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]

    return None


def has_complete_json_object(text: str) -> bool:
    start = text.find("{")
    if start == -1:
        return False

    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return True

    return False


class JsonStopOnComplete(StoppingCriteria):
    def __init__(self, tokenizer, prompt_length: int):
        self.tokenizer = tokenizer
        self.prompt_length = prompt_length

    def __call__(self, input_ids, scores, **kwargs) -> bool:
        generated_ids = input_ids[0][self.prompt_length:]
        text = self.tokenizer.decode(generated_ids, skip_special_tokens=True)
        return has_complete_json_object(text)


def normalize_unified_result(obj: Dict[str, Any]) -> Dict[str, Any]:
    if not isinstance(obj, dict):
        return {"type": "invalid"}

    result_type = obj.get("type", "invalid")
    if not isinstance(result_type, str):
        return {"type": "invalid"}

    result_type = result_type.strip().lower()

    if result_type == "direct_command":
        device_v = obj.get("device")
        action_v = obj.get("action")
        value_v = obj.get("value")

        if not isinstance(device_v, str) or not isinstance(action_v, str):
            return {"type": "invalid"}

        device_v = device_v.strip().lower()
        action_v = action_v.strip().lower()

        if "|" in device_v or "|" in action_v:
            return {"type": "invalid"}

        if value_v in ["null", "None", "none", "NULL"]:
            value_v = None

        if isinstance(value_v, str):
            value_v = value_v.strip().replace("%", "")
            if re.fullmatch(r"\d{1,3}", value_v):
                value_v = int(value_v)

        return {
            "type": "direct_command",
            "device": device_v,
            "action": action_v,
            "value": value_v,
        }

    if result_type == "needs_clarification":
        question = obj.get("question", "")
        options = obj.get("options", [])

        if not isinstance(question, str):
            return {"type": "invalid"}

        if not isinstance(options, list):
            return {"type": "invalid"}

        options = [str(x).strip() for x in options if str(x).strip()]
        if len(options) == 0:
            return {"type": "invalid"}

        return {
            "type": "needs_clarification",
            "question": question.strip(),
            "options": options,
        }

    if result_type == "general_qa":
        answer = obj.get("answer", "")
        if not isinstance(answer, str):
            return {"type": "invalid"}
        return {
            "type": "general_qa",
            "answer": answer.strip(),
        }

    return {"type": "invalid"}


def llm_generate_json(system_prompt: str, user_prompt: str, max_new_tokens: int = 96, verbose: bool = False):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_length = inputs["input_ids"].shape[1]

    stopping_criteria = StoppingCriteriaList([
        JsonStopOnComplete(tokenizer, prompt_length)
    ])

    start_time = time.perf_counter()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
            stopping_criteria=stopping_criteria
        )

    latency_ms = (time.perf_counter() - start_time) * 1000

    generated_ids = outputs[0][prompt_length:]
    raw_output = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    if verbose:
        print("Raw model output:", raw_output)
        print(f"LLM generation latency: {latency_ms:.3f} ms")

    json_str = extract_first_json_object(raw_output)
    if json_str is None:
        return {"type": "invalid"}, raw_output, latency_ms

    try:
        parsed = json.loads(json_str)
    except json.JSONDecodeError:
        return {"type": "invalid"}, raw_output, latency_ms

    return normalize_unified_result(parsed), raw_output, latency_ms


def llm_parse_unified(text: str, verbose: bool = False):
    return llm_generate_json(
        UNIFIED_SYSTEM_PROMPT,
        f'Text: "{text}"\nReturn JSON only.',
        max_new_tokens=96,
        verbose=verbose
    )


def llm_resolve_followup(original_text: str, question: str, options: List[str], user_reply: str, verbose: bool = False):
    options_text = "\n".join([f"- {opt}" for opt in options])

    user_prompt = f"""
Original request: "{original_text}"
Clarification question: "{question}"
Available options:
{options_text}
User reply: "{user_reply}"

Return JSON only.
""".strip()

    return llm_generate_json(
        FOLLOWUP_RESOLUTION_SYSTEM_PROMPT,
        user_prompt,
        max_new_tokens=96,
        verbose=verbose
    )

print("Unified parser loaded.")

Loading unified multi-task LLM...
Device: cpu


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.


Unified parser loaded.


In [29]:
# Section 6: 音频参数与录音函数

SAMPLE_RATE = 16000
CHANNELS = 1
DTYPE = "float32"
ENERGY_THRESHOLD = 0.01

def record_audio(duration_sec=3, sample_rate=SAMPLE_RATE, channels=CHANNELS, dtype=DTYPE):
    print(f"Recording {duration_sec} seconds...")
    audio = sd.rec(
        int(duration_sec * sample_rate),
        samplerate=sample_rate,
        channels=channels,
        dtype=dtype
    )
    sd.wait()
    return audio

def save_audio(audio, filename="temp_audio.wav", sample_rate=SAMPLE_RATE):
    sf.write(filename, audio, sample_rate)
    return filename

def audio_energy(audio):
    return float(np.mean(np.abs(audio)))

print("Audio functions loaded.")

Audio functions loaded.


In [30]:
# Section 7: 加载 STT 模型
# 用途：使用 faster-whisper 把音频 wav 转成文本

import io

WHISPER_MODEL_SIZE = "tiny.en"
WHISPER_DEVICE = "cpu"
WHISPER_COMPUTE_TYPE = "int8"

print("Loading STT model...")
stt_model = WhisperModel(
    WHISPER_MODEL_SIZE,
    device=WHISPER_DEVICE,
    compute_type=WHISPER_COMPUTE_TYPE
)
print("STT model loaded.")

def transcribe_audio_file(audio_path: str) -> str:
    segments, info = stt_model.transcribe(audio_path, beam_size=1)
    text = " ".join(seg.text.strip() for seg in segments).strip()
    return text

def transcribe_audio_numpy(audio: np.ndarray) -> str:
    """Transcribe directly from in-memory numpy array, avoiding disk I/O."""
    buf = io.BytesIO()
    sf.write(buf, audio, SAMPLE_RATE, format="wav")
    buf.seek(0)
    segments, _ = stt_model.transcribe(buf, beam_size=1)
    return " ".join(seg.text.strip() for seg in segments).strip()

Loading STT model...
STT model loaded.


In [31]:
# Section 7.5: TTS
# 用途：把系统回复通过扬声器播报出来

tts_engine = pyttsx3.init()
tts_engine.setProperty("rate", 170)

def speak_text(text: str, verbose: bool = True):
    if verbose:
        print("[TTS]", text)
    tts_engine.say(text)
    tts_engine.runAndWait()

def safe_speak_text(text: str, verbose: bool = True):
    if verbose:
        print("[TTS]", text)
    try:
        speak_text(text, verbose=False)
    except Exception as e:
        print("[TTS ERROR]", e)

print("TTS loaded.")

TTS loaded.


In [32]:
# Section 8: 带多轮对话状态的统一文本处理逻辑

DIALOGUE_STATE = {
    "pending_clarification": False,
    "original_text": None,
    "question": None,
    "options": None,
}

def reset_dialogue_state():
    DIALOGUE_STATE["pending_clarification"] = False
    DIALOGUE_STATE["original_text"] = None
    DIALOGUE_STATE["question"] = None
    DIALOGUE_STATE["options"] = None


# ---------- option 名称到自然语言描述的映射 ----------
OPTION_DISPLAY_MAP = {
    "close_window":         "Close the window",
    "raise_ac_temperature": "Raise the AC temperature",
    "lower_ac_temperature": "Lower the AC temperature",
    "turn_on_light":        "Turn on the light",
    "open_curtain":         "Open the curtain",
    "turn_off_light":       "Turn off the light",
    "dim_light":            "Dim the light",
    "rgb_cycle":            "Start RGB cycle mode",
    "turn_on_ac":           "Turn on the AC",
    "turn_off_ac":          "Turn off the AC",
    "close_curtain":        "Close the curtain",
    "open_window":          "Open the window",
}

def option_to_display(option: str) -> str:
    """将 option key 转为用户可读的描述；若无映射则做简单格式化。"""
    if option in OPTION_DISPLAY_MAP:
        return OPTION_DISPLAY_MAP[option]
    return option.replace("_", " ").capitalize()


def build_clarification_reply(question: str, options: list) -> str:
    """
    构造带编号选项的澄清回复，用于 TTS 和终端打印。
    示例输出:
      Would you like me to close the window or raise the AC temperature?
      Here are your options:
      Option 1: Close the window.
      Option 2: Raise the AC temperature.
      Please say the option number or describe what you want.
    """
    lines = [question, "Here are your options:"]
    for idx, opt in enumerate(options, start=1):
        lines.append(f"Option {idx}: {option_to_display(opt)}.")
    lines.append("Please say the option number or describe what you want.")
    return " ".join(lines)


# ================================================================
# 间接请求语义安全网 (Indirect-Request Safety Net)
# ================================================================
# TinyLlama 1.1B 对间接请求的分类不稳定，经常把 "I feel cold" 这类
# 感受型请求错误地分类为 direct_command 或 general_qa。
# 安全网在 LLM 输出之后做一次基于关键词的检查：
#   - 如果用户输入匹配了间接请求模式 **且** LLM 没有正确输出
#     needs_clarification，就强制降级为 needs_clarification。
# 这不会影响明确的直接命令（如 "turn on the light"），因为
# 直接命令同时包含设备名 + 动作词，会被排除在外。

INDIRECT_PATTERNS = [
    {
        # 感觉冷: "I feel cold / chilly / freezing / a bit cold"
        "keywords": [r"\bcold\b", r"\bchilly\b", r"\bfreezing\b"],
        "question": "Would you like me to close the window or raise the AC temperature?",
        "options": ["close_window", "raise_ac_temperature"],
    },
    {
        # 感觉热: "I feel hot / warm / stuffy"
        "keywords": [r"\bhot\b", r"\bwarm\b", r"\bstuffy\b", r"\bsweating\b"],
        "question": "Would you like me to open the window or lower the AC temperature?",
        "options": ["open_window", "lower_ac_temperature"],
    },
    {
        # 光线暗: "it's dark / dim / can't see"
        "keywords": [r"\bdark\b", r"\bdim\b", r"\bcan'?t see\b"],
        "question": "Would you like me to turn on the light or open the curtain?",
        "options": ["turn_on_light", "open_curtain"],
    },
    {
        # 光线太亮: "too bright / glare"
        "keywords": [r"\bbright\b", r"\bglare\b", r"\btoo much light\b"],
        "question": "Would you like me to turn off the light or close the curtain?",
        "options": ["turn_off_light", "close_curtain"],
    },
    {
        # 氛围类: "lively / boring / dull / vibe / mood"
        "keywords": [r"\blively\b", r"\bboring\b", r"\bdull\b", r"\bvibe\b", r"\bmood\b"],
        "question": "Would you like me to turn on the RGB cycle or open the curtain?",
        "options": ["rgb_cycle", "open_curtain"],
    },
    {
        # 表达不满（针对灯）: "fuck / damn / hate / annoying + light"
        "keywords": [r"(?:annoy|fuck|damn|hate).*\blight\b", r"\blight\b.*(?:annoy|fuck|damn|hate)"],
        "question": "Would you like me to turn off the light or dim it?",
        "options": ["turn_off_light", "dim_light"],
    },
]

# 明确的直接命令动作词 —— 如果输入同时包含这些词，说明用户意图明确，
# 不应被安全网拦截。例如 "turn on the light" 包含 "turn on"，属于直接命令。
EXPLICIT_ACTION_WORDS = [
    r"\bturn on\b", r"\bturn off\b", r"\bswitch on\b", r"\bswitch off\b",
    r"\bopen\b", r"\bclose\b", r"\bset\b", r"\brgb\b",
]

def check_indirect_request(text: str):
    """
    检测用户输入是否为间接/感受类请求。
    返回 (question, options) 或 None。
    如果输入同时包含明确的动作词（如 "turn on"），则不触发安全网。
    """
    t = text.lower()

    # 如果用户说了明确的动作词，说明是直接命令，不拦截
    for action_pat in EXPLICIT_ACTION_WORDS:
        if re.search(action_pat, t):
            return None

    # 检查是否匹配间接请求模式
    for pattern in INDIRECT_PATTERNS:
        for kw in pattern["keywords"]:
            if re.search(kw, t):
                return pattern["question"], pattern["options"]

    return None


def handle_transcribed_text(text: str, verbose: bool = True) -> Dict[str, Any]:
    text = text.strip()

    if verbose:
        print("STT text:", text)

    # 如果当前正在等待用户回答澄清问题，就不强制要求再说 Nova
    if DIALOGUE_STATE["pending_clarification"]:
        if not text:
            return {
                "prefilter_passed": True,
                "semantic": {"type": "invalid"},
                "valid": False,
                "reason": "empty_clarification_reply",
                "execution": "SKIPPED",
            }

        semantic, raw_output, llm_latency_ms = llm_resolve_followup(
            DIALOGUE_STATE["original_text"],
            DIALOGUE_STATE["question"],
            DIALOGUE_STATE["options"],
            text,
            verbose=verbose
        )

        if semantic["type"] == "direct_command":
            cmd = {
                "device": semantic["device"],
                "action": semantic["action"],
                "value": semantic["value"],
            }

            is_valid, reason = validate_command(cmd)

            if is_valid:
                execution = execute_command(cmd)
                spoken_reply = build_execution_reply(cmd)
                safe_speak_text(spoken_reply)
                reset_dialogue_state()
                return {
                    "prefilter_passed": True,
                    "semantic": semantic,
                    "valid": True,
                    "reason": reason,
                    "execution": execution,
                    "spoken_reply": spoken_reply,
                    "llm_latency_ms": round(llm_latency_ms, 3),
                }

            reply = "Sorry, I could not resolve that action safely."
            safe_speak_text(reply)
            reset_dialogue_state()
            return {
                "prefilter_passed": True,
                "semantic": semantic,
                "valid": False,
                "reason": reason,
                "execution": "SKIPPED",
                "spoken_reply": reply,
                "llm_latency_ms": round(llm_latency_ms, 3),
            }

        reply = "Sorry, I didn't catch your choice. Please answer again."
        safe_speak_text(reply)
        return {
            "prefilter_passed": True,
            "semantic": {"type": "invalid"},
            "valid": False,
            "reason": "clarification_not_resolved",
            "execution": "SKIPPED",
            "spoken_reply": reply,
            "llm_latency_ms": round(llm_latency_ms, 3),
        }

    # 正常新请求必须带 Nova
    if not text:
        return {
            "prefilter_passed": False,
            "semantic": {"type": "invalid"},
            "valid": False,
            "reason": "empty_text",
            "execution": "SKIPPED",
        }

    if not contains_assistant_name(text):
        if verbose:
            print("Assistant name not detected. Skip.")
        return {
            "prefilter_passed": False,
            "semantic": {"type": "invalid"},
            "valid": False,
            "reason": "assistant_name_not_detected",
            "execution": "SKIPPED",
        }

    semantic, raw_output, llm_latency_ms = llm_parse_unified(text, verbose=verbose)

    # ---- 语义安全网：检测间接请求被 LLM 错误分类的情况 ----
    # 当 LLM 输出不是 needs_clarification，但输入实际是间接请求时，
    # 强制覆盖为 needs_clarification。
    if semantic["type"] != "needs_clarification":
        indirect = check_indirect_request(text)
        if indirect is not None:
            override_question, override_options = indirect
            if verbose:
                print(f"[Safety Net] LLM returned '{semantic['type']}' but input "
                      f"contains indirect keywords. Overriding to needs_clarification.")
            semantic = {
                "type": "needs_clarification",
                "question": override_question,
                "options": override_options,
            }

    if semantic["type"] == "direct_command":
        cmd = {
            "device": semantic["device"],
            "action": semantic["action"],
            "value": semantic["value"],
        }

        is_valid, reason = validate_command(cmd)

        if is_valid:
            execution = execute_command(cmd)
            spoken_reply = build_execution_reply(cmd)
            safe_speak_text(spoken_reply)
            return {
                "prefilter_passed": True,
                "semantic": semantic,
                "valid": True,
                "reason": reason,
                "execution": execution,
                "spoken_reply": spoken_reply,
                "llm_latency_ms": round(llm_latency_ms, 3),
            }

        return {
            "prefilter_passed": True,
            "semantic": semantic,
            "valid": False,
            "reason": reason,
            "execution": "SKIPPED",
            "spoken_reply": None,
            "llm_latency_ms": round(llm_latency_ms, 3),
        }

    if semantic["type"] == "needs_clarification":
        DIALOGUE_STATE["pending_clarification"] = True
        DIALOGUE_STATE["original_text"] = text
        DIALOGUE_STATE["question"] = semantic["question"]
        DIALOGUE_STATE["options"] = semantic["options"]

        # 构造带编号选项的澄清回复
        clarification_reply = build_clarification_reply(
            semantic["question"], semantic["options"]
        )

        if verbose:
            print("[Clarification] Options presented to user:")
            for idx, opt in enumerate(semantic["options"], start=1):
                print(f"  [{idx}] {option_to_display(opt)}")

        safe_speak_text(clarification_reply)
        return {
            "prefilter_passed": True,
            "semantic": semantic,
            "valid": True,
            "reason": "clarification_requested",
            "execution": "PENDING_USER_REPLY",
            "spoken_reply": clarification_reply,
            "llm_latency_ms": round(llm_latency_ms, 3),
        }

    if semantic["type"] == "general_qa":
        answer = semantic["answer"]
        safe_speak_text(answer)
        return {
            "prefilter_passed": True,
            "semantic": semantic,
            "valid": True,
            "reason": "general_qa_answered",
            "execution": "NO_DEVICE_ACTION",
            "spoken_reply": answer,
            "llm_latency_ms": round(llm_latency_ms, 3),
        }

    reply = "Sorry, I didn't understand that."
    safe_speak_text(reply)
    return {
        "prefilter_passed": True,
        "semantic": {"type": "invalid"},
        "valid": False,
        "reason": "invalid_semantic_result",
        "execution": "SKIPPED",
        "spoken_reply": reply,
        "llm_latency_ms": round(llm_latency_ms, 3),
    }

In [33]:
# Section 9: 文本级测试

text_tests = [
    "Hello Nova, turn on the light.",
    "Nova, turn off the light.",
    "Nova, I feel a little bit cold.",
    "Nova, it's a bit dark.",
    "Nova, how can I eat an apple?",
    "How are you today?",
]

for text in text_tests:
    print("=" * 80)
    result = handle_transcribed_text(text, verbose=True)
    print(result)

Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


STT text: Hello Nova, turn on the light.


Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Raw model output: Input: Hello Nova, turn on the light.
Output: {"type":"direct_command","device":"light","action":"turn_on","value":null}
LLM generation latency: 10120.110 ms
[TTS] Okay, I turned on the light.
{'prefilter_passed': True, 'semantic': {'type': 'direct_command', 'device': 'light', 'action': 'turn_on', 'value': None}, 'valid': True, 'reason': 'ok', 'execution': 'LIGHT -> ON', 'spoken_reply': 'Okay, I turned on the light.', 'llm_latency_ms': 10120.11}
STT text: Nova, turn off the light.


Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Raw model output: Input: Nova, turn off the light.
Output: {"type":"direct_command","device":"light","action":"turn_off","value":null}
LLM generation latency: 8471.215 ms
[TTS] Okay, I turned off the light.
{'prefilter_passed': True, 'semantic': {'type': 'direct_command', 'device': 'light', 'action': 'turn_off', 'value': None}, 'valid': True, 'reason': 'ok', 'execution': 'LIGHT -> OFF', 'spoken_reply': 'Okay, I turned off the light.', 'llm_latency_ms': 8471.215}
STT text: Nova, I feel a little bit cold.


Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Raw model output: Input: Nova, I feel a little bit cold.
Output: {"type":"direct_command","device":"ac","action":"set_temperature","value":24}
LLM generation latency: 9440.885 ms
[Safety Net] LLM returned 'direct_command' but input contains indirect keywords. Overriding to needs_clarification.
[Clarification] Options presented to user:
  [1] Close the window
  [2] Raise the AC temperature
[TTS] Would you like me to close the window or raise the AC temperature? Here are your options: Option 1: Close the window. Option 2: Raise the AC temperature. Please say the option number or describe what you want.
{'prefilter_passed': True, 'semantic': {'type': 'needs_clarification', 'question': 'Would you like me to close the window or raise the AC temperature?', 'options': ['close_window', 'raise_ac_temperature']}, 'valid': True, 'reason': 'clarification_requested', 'execution': 'PENDING_USER_REPLY', 'spoken_reply': 'Would you like me to close the window or raise the AC temperature? Here are your 

Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Raw model output: Response:
```
{
  "type": "direct_command",
  "device": "light",
  "action": "turn_on",
  "value": "false"
}
LLM generation latency: 11715.116 ms
[TTS] Sorry, I could not resolve that action safely.
{'prefilter_passed': True, 'semantic': {'type': 'direct_command', 'device': 'light', 'action': 'turn_on', 'value': 'false'}, 'valid': False, 'reason': 'value_must_be_null', 'execution': 'SKIPPED', 'spoken_reply': 'Sorry, I could not resolve that action safely.', 'llm_latency_ms': 11715.116}
STT text: Nova, how can I eat an apple?
Raw model output: Input: Nova, how can I eat an apple?
Output: {"type":"general_qa","answer":"Wash it first, then eat it."}
LLM generation latency: 8653.203 ms
[TTS] Wash it first, then eat it.
{'prefilter_passed': True, 'semantic': {'type': 'general_qa', 'answer': 'Wash it first, then eat it.'}, 'valid': True, 'reason': 'general_qa_answered', 'execution': 'NO_DEVICE_ACTION', 'spoken_reply': 'Wash it first, then eat it.', 'llm_latency_ms': 8653.20

In [ ]:
# Section 10: 单次完整测试
# 用途：单次录音，完整跑通录音 -> STT -> Nova预筛选 -> unified parser -> validator/executor/TTS

CHUNK_DURATION = 3

def run_one_round():
    print("[1/6] Start recording...")
    audio = record_audio(duration_sec=CHUNK_DURATION)

    print("[2/6] Recording finished.")
    energy = audio_energy(audio)
    print(f"[2/6] Audio energy: {energy:.6f}")

    if energy < ENERGY_THRESHOLD:
        print("[STOP] Too quiet, skipped.")
        return None

    print("[3/6] Transcribing audio (in-memory)...")

    print("[4/6] Running STT...")
    start_stt = time.perf_counter()
    text = transcribe_audio_numpy(audio)
    stt_latency_ms = (time.perf_counter() - start_stt) * 1000

    print(f"[4/6] Transcribed text: {text}")
    print(f"[4/6] STT latency: {stt_latency_ms:.3f} ms")

    if not text:
        print("[STOP] Empty transcription.")
        return {
            "prefilter_passed": False,
            "semantic": {"type": "invalid"},
            "valid": False,
            "reason": "empty_text",
            "execution": "SKIPPED",
            "stt_latency_ms": round(stt_latency_ms, 3),
        }

    print("[5/6] Running unified pipeline...")
    start_pipeline = time.perf_counter()
    result = handle_transcribed_text(text, verbose=True)
    pipeline_latency_ms = (time.perf_counter() - start_pipeline) * 1000

    print(f"[5/6] Pipeline latency: {pipeline_latency_ms:.3f} ms")
    print("[6/6] Done.")

    if isinstance(result, dict):
        result["stt_latency_ms"] = round(stt_latency_ms, 3)
        result["pipeline_latency_ms"] = round(pipeline_latency_ms, 3)

    return result

In [ ]:
result = run_one_round()
print(result)

In [14]:
# Section 11: 持续监听参数

FRAME_DURATION = 0.1
FRAME_SAMPLES = int(SAMPLE_RATE * FRAME_DURATION)

SILENCE_SECONDS = 0.5
MIN_SPEECH_SECONDS = 0.3
MAX_UTTERANCE_SECONDS = 8
PRE_ROLL_SECONDS = 0.3

SILENCE_FRAMES = int(SILENCE_SECONDS / FRAME_DURATION)
PRE_ROLL_FRAMES = int(PRE_ROLL_SECONDS / FRAME_DURATION)
MAX_FRAMES = int(MAX_UTTERANCE_SECONDS / FRAME_DURATION)

print("Listener parameters loaded.")

Listener parameters loaded.


In [15]:
# Section 12: 从持续音频流中截取一句完整话语

def frame_energy(frame: np.ndarray) -> float:
    return float(np.mean(np.abs(frame)))

def collect_one_utterance_from_stream(
    stream,
    energy_threshold=ENERGY_THRESHOLD,
    frame_samples=FRAME_SAMPLES,
    silence_frames=SILENCE_FRAMES,
    min_speech_seconds=MIN_SPEECH_SECONDS,
    max_frames=MAX_FRAMES,
    pre_roll_frames=PRE_ROLL_FRAMES,
):
    pre_buffer = deque(maxlen=pre_roll_frames)
    collected_frames = []

    speech_started = False
    silence_count = 0
    speech_frame_count = 0

    while True:
        frame, overflowed = stream.read(frame_samples)
        frame = frame.copy()
        energy = frame_energy(frame)

        if not speech_started:
            pre_buffer.append(frame)

            if energy >= energy_threshold:
                speech_started = True
                collected_frames.extend(list(pre_buffer))
                collected_frames.append(frame)
                speech_frame_count += 1
                silence_count = 0
        else:
            collected_frames.append(frame)

            if energy >= energy_threshold:
                speech_frame_count += 1
                silence_count = 0
            else:
                silence_count += 1

            if silence_count >= silence_frames:
                break

            if len(collected_frames) >= max_frames:
                break

    if not speech_started:
        return None

    total_speech_seconds = speech_frame_count * FRAME_DURATION
    if total_speech_seconds < min_speech_seconds:
        return None

    audio = np.concatenate(collected_frames, axis=0)
    return audio

In [ ]:
# Section 13: 处理单句音频

def process_utterance_audio(audio, verbose: bool = True):
    start_stt = time.perf_counter()
    text = transcribe_audio_numpy(audio)
    stt_latency_ms = (time.perf_counter() - start_stt) * 1000

    if verbose:
        print("[STT] Text:", text)
        print(f"[STT] Latency: {stt_latency_ms:.3f} ms")

    result = handle_transcribed_text(text, verbose=verbose)

    return {
        "text": text,
        "action_taken": "processed_after_name_prefilter" if result.get("prefilter_passed") else "skipped",
        "result": result,
        "stt_latency_ms": round(stt_latency_ms, 3),
    }

In [17]:
# Section 14: 持续监听主循环

def continuous_listen_loop():
    print("Continuous listening started.")
    print("Say 'Nova' to start a new request.")
    print("If Nova asks a follow-up question, you can answer directly without saying Nova again.")
    print("Press Ctrl+C to stop.\n")

    try:
        with sd.InputStream(
            samplerate=SAMPLE_RATE,
            channels=CHANNELS,
            dtype=DTYPE,
            blocksize=FRAME_SAMPLES,
        ) as stream:

            while True:
                print("\n[Listener] Waiting for utterance...")

                audio = collect_one_utterance_from_stream(
                    stream,
                    energy_threshold=ENERGY_THRESHOLD,
                )

                if audio is None:
                    print("[Listener] No valid utterance captured.")
                    continue

                print("[Listener] Utterance captured. Processing...")

                output = process_utterance_audio(audio, verbose=True)
                print("[Output]", output)
                print("-" * 80)

    except KeyboardInterrupt:
        print("\nContinuous listening stopped.")

In [21]:
continuous_listen_loop()

Continuous listening started.
Say 'Nova' to start a new request.
If Nova asks a follow-up question, you can answer directly without saying Nova again.
Press Ctrl+C to stop.


[Listener] Waiting for utterance...
[Listener] No valid utterance captured.

[Listener] Waiting for utterance...
[Listener] No valid utterance captured.

[Listener] Waiting for utterance...
[Listener] No valid utterance captured.

[Listener] Waiting for utterance...
[Listener] Utterance captured. Processing...
[STT] Text: 
[STT] Latency: 445.387 ms
STT text: 
[Output] {'text': '', 'action_taken': 'skipped', 'result': {'prefilter_passed': False, 'semantic': {'type': 'invalid'}, 'valid': False, 'reason': 'empty_text', 'execution': 'SKIPPED'}, 'stt_latency_ms': 445.387}
--------------------------------------------------------------------------------

[Listener] Waiting for utterance...
[Listener] Utterance captured. Processing...
[STT] Text: No no, Bob, I feel cold.
[STT] Latency: 511.411 ms
STT text: No no, Bob, I f

In [22]:
# Section 15: 离线批量测试集
# 用途：离线测试 nova 预筛选 + LLM 解析效果

# Section 15: 离线批量测试集（统一输出版本）
# 用途：离线测试 Nova 预筛选 + unified semantic parser

test_cases = [
    {
        "input": "Nova, I feel a little bit cold.",
        "expected_type": "needs_clarification"
    },
    {
        "input": "Nova, it's a bit dark.",
        "expected_type": "needs_clarification"
    },
    {
        "input": "Nova, fuck this light.",
        "expected_type": "needs_clarification"
    },
    {
        "input": "Nova, make this room lively.",
        "expected_type": "needs_clarification"
    },
    {
        "input": "Nova, how can I eat an apple?",
        "expected_type": "general_qa"
    },
    {
        "input": "Nova, can I still eat this dish after one night in the fridge?",
        "expected_type": "general_qa"
    },
   
]

In [23]:
# Section 16: 批量评估（统一输出版本）

results = []

for item in test_cases:
    user_input = item["input"]
    expected = item.get("expected", None)
    expected_type = item.get("expected_type", None)

    start = time.perf_counter()
    result = handle_transcribed_text(user_input, verbose=False)
    latency_ms = (time.perf_counter() - start) * 1000

    semantic = result.get("semantic", None)

    # 情况 1：不含 Nova，应该被 prefilter 跳过
    if expected is None and expected_type is None:
        exact_match = (result["prefilter_passed"] == False)

    # 情况 2：要求精确 direct_command / invalid
    elif expected is not None:
        exact_match = (semantic == expected)

    # 情况 3：只检查类型
    else:
        exact_match = (
            isinstance(semantic, dict)
            and semantic.get("type") == expected_type
        )

    results.append({
        "input": user_input,
        "expected": json.dumps(expected) if expected is not None else (
            expected_type if expected_type is not None else "PREFILTER_SKIP"
        ),
        "semantic": json.dumps(semantic) if semantic is not None else "NONE",
        "prefilter_passed": result.get("prefilter_passed", False),
        "valid": result.get("valid", False),
        "reason": result.get("reason", ""),
        "execution": result.get("execution", "SKIPPED"),
        "exact_match": exact_match,
        "latency_ms": round(latency_ms, 3),
    })

df = pd.DataFrame(results)
df

Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[TTS] Would you like me to close the window or raise the AC temperature?


Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[TTS] Sorry, I could not resolve that action safely.


Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[TTS] You can ask Nova to make the room lively by saying 'make this room lively'.


Both `max_new_tokens` (=96) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[TTS] You can wash the apple first and then eat it.
[TTS] Would you like me to cook the dish or store it in the fridge?


,input,expected,semantic,prefilter_passed,valid,reason,execution,exact_match,latency_ms
0,"Nova, I feel a little bit cold.",needs_clarification,"{""type"": ""needs_clarification"", ""question"": ""W...",True,True,clarification_requested,PENDING_USER_REPLY,True,27330.007
1,"Nova, it's a bit dark.",needs_clarification,"{""type"": ""direct_command"", ""device"": ""light"", ...",True,False,value_must_be_null,SKIPPED,False,12785.564
2,"Nova, fuck this light.",needs_clarification,"{""type"": ""general_qa"", ""answer"": ""You can turn...",True,True,general_qa_answered,NO_DEVICE_ACTION,False,13875.505
3,"Nova, make this room lively.",needs_clarification,"{""type"": ""general_qa"", ""answer"": ""You can ask ...",True,True,general_qa_answered,NO_DEVICE_ACTION,False,16099.502
4,"Nova, how can I eat an apple?",general_qa,"{""type"": ""general_qa"", ""answer"": ""You can wash...",True,True,general_qa_answered,NO_DEVICE_ACTION,True,16281.406
5,"Nova, can I still eat this dish after one nigh...",general_qa,"{""type"": ""needs_clarification"", ""question"": ""W...",True,True,clarification_requested,PENDING_USER_REPLY,False,24041.896


In [24]:
# Section 16: 输出指标（统一输出版本）

print("\n===== Metrics =====")
print(f"Exact Match Accuracy: {df['exact_match'].mean():.2%}")
print(f"Avg Latency: {df['latency_ms'].mean():.3f} ms")


===== Metrics =====
Exact Match Accuracy: 33.33%
Avg Latency: 18402.313 ms
